# 08 · Geocode the NT school list

**Replaces** `geocode_schools.py`.

The NT Department of Education school list has no coordinates. Each school gets the coordinate
of its **locality** (suburb or community), looked up in a local gazetteer. This is
locality precision, not street precision, which fits the project rule that distance work here is triage.

**Reads**

| file | used for |
|---|---|
| `data/external/services/trimmed/school_list_trimmed.csv` | the 273 schools, their locality, region, sector and stage flags |
| `data/processed/medical_services.csv` (from nb 07) | suburb centroids |
| `data/processed/emergency_services.csv` (from nb 07) | suburb centroids |
| `data/raw/bushtel/Com_BushTel_Profile_CMC_2024.json` | 792 NT communities and their aliases |

**Writes** `data/processed/school_services.csv`.

**Fixes compared with the old script**

1. **Katherine schools were ~57 km out of place.** Suburb centroids were a plain mean, and the
   `Katherine` suburb contains clinics from Ngukurr, Minyerri and Barunga. Rows flagged in nb 07
   are now left out of centroids.
2. **Postcodes stuck to the locality** (`Wulagi 0812`, `Holtze 0829`) are stripped, and
   placeholders (`NT`, `tba`) are treated as missing.
3. **Darwin suburbs no longer collapse onto the Darwin centre.** Alawa, Anula, Nakara, Wanguri
   and others now use their own Wikipedia coordinates (cited per row) instead of the region fallback.
4. **More schools are resolved.** Two new tiers were added. The first tries a shorter locality
   (`Pickertaramoor Melville Island` → `Pickertaramoor`). The second finds a BushTel community
   named in the school's own name (`Nawarddeken Academy Manmoyi School` → Manmoyi,
   `Woolaning School` → Woolaning).
5. **New `school_geocode_precision` column** (`suburb` / `community` / `region` / `none`), so
   anything downstream can tell a real locality from a coarse fallback.
6. A school with no region is filled from other schools in the same locality.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# Walk up from wherever Jupyter was started until we find the repo root
# (the folder that holds data/raw), so the notebook runs from any cwd.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir())
INPUTS = ROOT / "data" / "external" / "services" / "trimmed"   # read-only inputs
PROCESSED = ROOT / "data" / "processed"                         # the only folder we write to

# NT bounding box used as a sanity check on every coordinate we write.
NT_LAT, NT_LON = (-26.1, -10.9), (128.9, 138.1)


def in_nt(df: pd.DataFrame) -> pd.Series:
    return df["latitude"].between(*NT_LAT) & df["longitude"].between(*NT_LON)


def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km (vectorised; any argument may be an array)."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371.0088 * np.arcsin(np.sqrt(a))

import json
import re

BUSHTEL_JSON = ROOT / "data" / "raw" / "bushtel" / "Com_BushTel_Profile_CMC_2024.json"


## 1. Load the school list

In [2]:
schools = pd.read_csv(INPUTS / "school_list_trimmed.csv")
assert schools["name"].is_unique
print(f"{len(schools)} schools")
schools.head()


273 schools


,source,name,school_locality,school_region,school_sector,school_remoteness,school_is_preschool,school_is_primary,school_is_middle,school_is_senior,school_is_faft
0,school,Acacia Hill School,Alice Springs,Central,Government,Urban,True,True,True,True,False
1,school,Adelaide River School,Adelaide River,Top End,Government,Remote Special Category,True,True,False,False,False
2,school,Alawa Pre School,Alawa,Darwin,NaN,NaN,True,False,False,False,False
3,school,Alawa Primary School,Alawa,Darwin,Government,Urban,True,True,False,False,False
4,school,Alcoota School,Engawala,Central,Government,Remote Category 2,True,True,True,True,False


## 2. Clean the locality field

The locality was parsed from a free-text address, so a few values still carry a postcode or are
placeholders rather than place names.

In [3]:
PLACEHOLDERS = {"", "NT", "N/A", "TBA"}


def norm(name) -> str:
    """Normalise a place name for matching: upper-case, single spaces, no trailing '(NT)'."""
    if name is None or (isinstance(name, float) and np.isnan(name)):
        return ""
    n = re.sub(r"\s+", " ", str(name).strip().upper())
    return re.sub(r"\s*\(NT\)$", "", n)


def clean_locality(raw):
    """Strip a trailing postcode and turn placeholders into None."""
    if pd.isna(raw):
        return None
    loc = re.sub(r"\s+\d{4}$", "", str(raw).strip())      # 'Wulagi 0812' -> 'Wulagi'
    return None if loc.upper() in PLACEHOLDERS else loc


schools["school_locality_raw"] = schools["school_locality"]
schools["school_locality"] = schools["school_locality"].map(clean_locality)

changed = schools[schools["school_locality_raw"].fillna("") != schools["school_locality"].fillna("")]
print(changed[["name", "school_locality_raw", "school_locality"]].to_string())


                                         name             school_locality_raw            school_locality
135                   Mapuru Yirralka College                         NT 0822                       None
162  Nawarddeken Academy Kabulwarnamyo School                             tba                       None
163      Nawarddeken Academy Kunmayali School                             tba                       None
164    Nawarddeken Academy Mamadawerre School                             tba                       None
165        Nawarddeken Academy Manmoyi School                             tba                       None
181           Northern Territory Music School                     Wulagi 0812                     Wulagi
217              School of Sport Education NT                              NT                       None
224              St Joseph's Catholic College                  Katherine 0850                  Katherine
230                           Stirling School  Wilora (

## 3. Build the gazetteer (place name → coordinate)

Sources in priority order (the first source to supply a name wins):

1. **Medical suburb centroids.** Mean of the facilities in that suburb, **excluding** rows nb 07
   flagged as `far_from_site`.
2. **Emergency suburb centroids.** Same rule.
3. **BushTel communities and their aliases.** One point per remote community.

Suburb centroids come first because a suburb's facilities sit inside the suburb. BushTel gives
one point per town, which for a big town like Alice Springs is less representative.

In [4]:
med = pd.read_csv(PROCESSED / "medical_services.csv")
emg = pd.read_csv(PROCESSED / "emergency_services.csv")

gaz: dict[str, tuple[float, float, str]] = {}      # NAME -> (lat, lon, source tag)


def add_gaz(name, lat, lon, source):
    key = norm(name)
    if key and pd.notna(lat) and pd.notna(lon) and key not in gaz:
        gaz[key] = (float(lat), float(lon), source)


# 3a/3b: suburb centroids from the facility files, leaving out mislabelled rows.
for df, suburb_col, flag_col, tag in [
    (med, "medical_suburb", "medical_suburb_far_from_site", "medical_suburb"),
    (emg, "emergency_suburb", "emergency_suburb_far_from_site", "emergency_suburb"),
]:
    good = df[~df[flag_col]]
    centroids = good.groupby(good[suburb_col].map(norm))[["latitude", "longitude"]].mean()
    for suburb, row in centroids.iterrows():
        add_gaz(suburb, row.latitude, row.longitude, tag)

# 3c: BushTel communities, then every recorded alias.
bushtel = json.loads(BUSHTEL_JSON.read_text(encoding="utf-8"))["features"]
bushtel_keys = set()
for feat in bushtel:
    p = feat["properties"]
    names = [p["community_name"]]
    aliases = p.get("community_aliases") or ""
    if "no aliases" not in aliases.lower():
        names += [a for a in re.split(r"[;,]", aliases) if a.strip()]
    for i, n in enumerate(names):
        add_gaz(n, p["latitude"], p["longitude"], "bushtel" if i == 0 else "bushtel_alias")
        bushtel_keys.add(norm(n))

print(f"gazetteer: {len(gaz)} names ->", pd.Series([v[2] for v in gaz.values()]).value_counts().to_dict())
print("KATHERINE now:", gaz["KATHERINE"])


gazetteer: 2156 names -> {'bushtel_alias': 1301, 'bushtel': 712, 'medical_suburb': 101, 'emergency_suburb': 42}
KATHERINE now: (-14.456183343590547, 132.26592077988724, 'medical_suburb')


## 4. Manual overrides (cited)

Darwin, Palmerston and rural-Darwin suburbs that have no facility in the medical/emergency files
and no BushTel entry. Each was looked up individually, and the source is stored with the row.
These **take priority** over the gazetteer. They're checked against the NT box below.

In [5]:
MANUAL_OVERRIDES = {
    # suburb          lat          lon          source
    "ALAWA":         (-12.380000, 130.873000, "wikipedia:Alawa,_Northern_Territory"),
    "ANULA":         (-12.392778, 130.890000, "wikipedia:Anula,_Northern_Territory"),
    "JINGILI":       (-12.386667, 130.875278, "wikipedia:Jingili,_Northern_Territory"),
    "NAKARA":        (-12.369444, 130.878056, "wikipedia:Nakara,_Northern_Territory"),
    "RAPID CREEK":   (-12.380833, 130.862222, "wikipedia:Rapid_Creek,_Northern_Territory"),
    "THE GARDENS":   (-12.448611, 130.836389, "wikipedia:The_Gardens,_Northern_Territory"),
    "LARRAKEYAH":    (-12.458611, 130.821667, "wikipedia:Larrakeyah,_Northern_Territory"),
    "WAGAMAN":       (-12.382500, 130.885000, "wikipedia:Wagaman,_Northern_Territory"),
    "WANGURI":       (-12.372500, 130.887778, "wikipedia:Wanguri,_Northern_Territory"),
    "WULAGI":        (-12.384444, 130.893056, "wikipedia:Wulagi,_Northern_Territory"),
    "HOLTZE":        (-12.461000, 130.984000, "wikipedia:Holtze,_Northern_Territory"),
    "FREDS PASS":    (-12.539836, 131.055036, "wikipedia:Freds_Pass,_Northern_Territory"),
    "MARLOW LAGOON": (-12.498405, 130.964515, "wikipedia:Marlow_Lagoon,_Northern_Territory"),
    "BEES CREEK":    (-12.555285, 131.059105, "wikipedia:Bees_Creek,_Northern_Territory"),
    "DRIVER":        (-12.485085, 130.973005, "wikipedia:Driver,_Northern_Territory"),
    "GIRRAWEEN":     (-12.524085, 131.092505, "wikipedia:Girraween,_Northern_Territory"),
    "GRAY":          (-12.492205, 130.978915, "wikipedia:Gray,_Northern_Territory"),
    "MOULDEN":       (-12.510000, 130.976000, "wikipedia:Moulden,_Northern_Territory"),
    "JOHNSTON":      (-12.486400, 131.009600, "wikipedia:Johnston,_Northern_Territory"),
    "DUNDEE BEACH":  (-12.733400, 130.383100, "wikipedia:Dundee_Beach,_Northern_Territory"),
    "MIDDLE POINT":  (-12.599000, 131.326100, "wikipedia:Middle_Point,_Northern_Territory"),
    "BELLAMACK":     (-12.521100, 130.980200, "latitude.to:Bellamack"),
    "TIPPERARY":     (-13.729839, 131.040459, "findlatitudeandlongitude.com:Tipperary_Station"),
}
for k, (lat, lon, _) in MANUAL_OVERRIDES.items():
    assert NT_LAT[0] <= lat <= NT_LAT[1] and NT_LON[0] <= lon <= NT_LON[1], k
print(f"{len(MANUAL_OVERRIDES)} manual overrides, all inside the NT box")


23 manual overrides, all inside the NT box


## 5. Match every school

Tiers, tried in order. The first hit wins and is recorded in `school_geocode_method`:

| tier | example | precision |
|---|---|---|
| `manual:<source>` | Alawa → Wikipedia | suburb |
| `exact:<gazetteer source>` | Adelaide River → medical suburb centroid | suburb / community |
| `prefix:<gazetteer source>` | *Pickertaramoor Melville Island* → Pickertaramoor | community |
| `name:bushtel` | *Nawarddeken Academy Manmoyi School* → Manmoyi | community |
| `region_fallback:<region>-><town>` | the region's main town | region (coarse!) |
| `unmatched` | no coordinate. Better than a wrong one | none |

**Fuzzy matching is deliberately not used.** Short Darwin suburb names (Alawa, Anula, Nakara)
kept fuzzy-matching remote BushTel aliases hundreds of km away. The `name:` tier only accepts a
**whole-word** BushTel community name of 5+ letters and prefers the longest match.

In [6]:
# Main town of each DoE region. Used only when nothing better matches.
# "Top End" has no single town of its own, so it's intentionally left out.
REGION_FALLBACK = {
    "DARWIN": "DARWIN", "PALMERSTON": "PALMERSTON", "CENTRAL": "ALICE SPRINGS",
    "BIG RIVERS": "KATHERINE", "EAST ARNHEM": "NHULUNBUY", "BARKLY": "TENNANT CREEK",
    "WEST ARNHEM": "JABIRU",
}
PRECISION = {"medical_suburb": "suburb", "emergency_suburb": "suburb",
             "bushtel": "community", "bushtel_alias": "community"}

# BushTel names long enough to search for inside a school's name, longest first.
NAME_KEYS = sorted((k for k in bushtel_keys if len(k) >= 5), key=len, reverse=True)


def match(name, locality, region):
    """Return (lat, lon, method, precision) for one school."""
    key = norm(locality)
    if key:
        # Tier 1: cited manual coordinate.
        if key in MANUAL_OVERRIDES:
            lat, lon, src = MANUAL_OVERRIDES[key]
            return lat, lon, f"manual:{src}", "suburb"
        # Tier 2: exact gazetteer hit.
        if key in gaz:
            lat, lon, src = gaz[key]
            return lat, lon, f"exact:{src}", PRECISION[src]
        # Tier 3: drop trailing words ("PICKERTARAMOOR MELVILLE ISLAND" -> "PICKERTARAMOOR").
        words = key.split()
        for n in range(len(words) - 1, 0, -1):
            shorter = " ".join(words[:n])
            if len(shorter) >= 5 and shorter in gaz:
                lat, lon, src = gaz[shorter]
                return lat, lon, f"prefix:{src}", PRECISION[src]
    # Tier 4: a BushTel community named in the school's own name.
    upper_name = norm(name)
    for k in NAME_KEYS:
        if re.search(rf"\b{re.escape(k)}\b", upper_name):
            lat, lon, src = gaz[k]
            return lat, lon, "name:bushtel", "community"
    # Tier 5: coarse regional fallback.
    town = REGION_FALLBACK.get(norm(region))
    if town and town in gaz:
        lat, lon, _ = gaz[town]
        return lat, lon, f"region_fallback:{region}->{town}", "region"
    return None, None, "unmatched", "none"


matched = schools.apply(lambda r: match(r["name"], r["school_locality"], r["school_region"]), axis=1)
schools[["latitude", "longitude", "school_geocode_method", "school_geocode_precision"]] = pd.DataFrame(
    matched.tolist(), index=schools.index)

print(schools["school_geocode_precision"].value_counts().to_string(), "\n")
print(schools["school_geocode_method"].str.split(":").str[0].value_counts().to_string())


school_geocode_precision
suburb       229
community     42
none           1
region         1 

school_geocode_method
exact              216
manual              44
name                 7
prefix               4
unmatched            1
region_fallback      1


### 5a. Review the matches that aren't a plain exact hit

In [7]:
review = schools[~schools["school_geocode_method"].str.startswith(("exact", "manual"))]
print(review[["name", "school_locality", "school_geocode_method", "latitude", "longitude"]].to_string())


                                         name                 school_locality           school_geocode_method   latitude   longitude
13                           Amanbidji School                       Amanbidgi                    name:bushtel -16.428060  129.618900
22                            Arlparra School                          Utopia                    name:bushtel -22.087790  134.774760
55                             Dhupuma Barker     Gunyangara East Arnhem Land           prefix:medical_suburb -12.220892  136.703512
135                   Mapuru Yirralka College                            None                    name:bushtel -12.253180  135.437790
155                         Mulga Bore School            Mulga Bore Community                  prefix:bushtel -22.451390  134.209440
162  Nawarddeken Academy Kabulwarnamyo School                            None                    name:bushtel -12.764860  133.844720
163      Nawarddeken Academy Kunmayali School                        

## 6. Fill a missing region from schools in the same locality

One school has no region. If every other school in its locality shares one region, that region is
used. Otherwise the gap stays.

In [8]:
locality_region = (schools.dropna(subset=["school_region"])
                   .groupby(schools["school_locality"].map(norm))["school_region"]
                   .agg(lambda s: s.iloc[0] if s.nunique() == 1 else None))
missing = schools["school_region"].isna()
schools.loc[missing, "school_region"] = schools.loc[missing, "school_locality"].map(norm).map(locality_region)
print(schools.loc[missing, ["name", "school_locality", "school_region"]].to_string())


                                                     name school_locality school_region
226  St Joseph's Catholic Flexible Learning Centre Darwin        Berrimah        Darwin


## 7. Validate

* every coordinate we assigned is inside the NT
* Katherine schools now sit in Katherine (within 5 km of the Katherine Hospital)
* no Darwin school is still on the Darwin region fallback when its suburb is known

In [9]:
has_xy = schools["latitude"].notna()
assert in_nt(schools[has_xy]).all()

kath = schools[schools["school_locality"].map(norm) == "KATHERINE"]
hosp = med[med["name"].str.startswith("Katherine Hospital")].iloc[0]
kath_km = haversine_km(kath["latitude"], kath["longitude"], hosp.latitude, hosp.longitude)
assert (kath_km < 5).all(), kath_km
print(f"{len(kath)} Katherine schools, max {kath_km.max():.1f} km from Katherine Hospital")

print(f"geocoded {int(has_xy.sum())}/{len(schools)} schools ({has_xy.mean():.1%})")
print("still unmatched:", schools.loc[~has_xy, ["name", "school_locality_raw"]].values.tolist())
print("\nlargest groups of schools sharing one point (expected in big towns, locality precision):")
print(schools[has_xy].groupby(["school_locality"]).size().sort_values(ascending=False).head(5).to_string())


9 Katherine schools, max 2.0 km from Katherine Hospital
geocoded 272/273 schools (99.6%)
still unmatched: [['Nawarddeken Academy Kunmayali School', 'tba']]

largest groups of schools sharing one point (expected in big towns, locality precision):
school_locality
Alice Springs     27
Katherine          9
Karama             6
Katherine East     6
Berrimah           4


## 8. Write output

In [10]:
OUT_COLS = ["source", "name", "latitude", "longitude", "school_locality", "school_geocode_method",
            "school_geocode_precision", "school_region", "school_sector", "school_remoteness",
            "school_is_preschool", "school_is_primary", "school_is_middle", "school_is_senior",
            "school_is_faft"]
schools[OUT_COLS].to_csv(PROCESSED / "school_services.csv", index=False)
print(f"wrote school_services.csv ({len(schools)} rows)")


wrote school_services.csv (273 rows)


### Known gaps (not fixed here, on purpose)

* **Locality precision only.** Every school in a town shares that town's point (e.g. all Alice
  Springs schools). The raw list's street addresses would fix this but aren't in the repo.
* **`school_sector` is blank for ~60 rows**, all government pre-schools, and
  **`school_remoteness` is blank for non-government schools**. Both come from the source data.
  The NTG remoteness definition only covers government schools.